# YOLO Fruit Detection Training - Kaggle Version

Train YOLO v8 model for multi-class fruit detection (Mango, Grapefruit, Guava, Orange)

**Hardware Requirements:**
- GPU: P100 or better
- RAM: 16GB+ recommended

**Dataset:**
- Add your dataset as Kaggle Dataset input
- Path will be: `/kaggle/input/your-dataset-name/`

In [ ]:
# Install Required Packages for Kaggle
!pip install ultralytics

print("✅ Packages installed successfully!")

In [ ]:
# Import Required Libraries
import os
import torch
from ultralytics import YOLO
import yaml
from pathlib import Path

print("✅ Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

In [ ]:
# Check Kaggle Environment and Dataset
print("🔍 Kaggle Environment Check:")
print(f"Working directory: {os.getcwd()}")
print(f"Available datasets: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else 'No input datasets'}")

# List all files in input directory to find your dataset
if os.path.exists('/kaggle/input'):
    for item in os.listdir('/kaggle/input'):
        item_path = f'/kaggle/input/{item}'
        if os.path.isdir(item_path):
            print(f"📁 Dataset: {item}")
            print(f"   Contents: {os.listdir(item_path)[:10]}...")  # Show first 10 items

In [ ]:
# Verify Dataset Structure
# ⚠️ IMPORTANT: Replace 'your-dataset-name' with your actual Kaggle dataset name
# Example: If your dataset is named 'multi-fruit-detection', use that name

DATASET_NAME = "your-dataset-name"  # 🔄 CHANGE THIS to your uploaded dataset name
dataset_path = Path(f'/kaggle/input/{DATASET_NAME}/multi_fruit_detection')

print(f"🔍 Looking for dataset at: {dataset_path}")

# If dataset is directly in input folder (not in subfolder)
if not dataset_path.exists():
    dataset_path = Path(f'/kaggle/input/{DATASET_NAME}')
    print(f"🔍 Trying alternative path: {dataset_path}")

# Quick check for required structure
if not dataset_path.exists():
    print("❌ Dataset not found!")
    print("Please check:")
    print("1. Dataset is uploaded to Kaggle")
    print("2. Dataset is added as input to this notebook")
    print("3. DATASET_NAME variable above matches your dataset name")
else:
    print("✅ Dataset directory found!")
    
    required_dirs = ['train/images', 'train/labels', 'val/images', 'val/labels']
    all_exist = all((dataset_path / d).exists() for d in required_dirs)
    
    if all_exist:
        print("✅ All required directories present!")
        
        # Count images for verification
        train_images = len(list((dataset_path / 'train/images').glob('*.jpg')))
        val_images = len(list((dataset_path / 'val/images').glob('*.jpg')))
        
        print(f"📊 Dataset Summary:")
        print(f"   Train images: {train_images}")
        print(f"   Val images: {val_images}")
        print(f"   Total: {train_images + val_images}")
    else:
        print("⚠️ Some required directories are missing!")
        for d in required_dirs:
            status = "✅" if (dataset_path / d).exists() else "❌"
            print(f"{status} {d}")

In [ ]:
# Create data.yaml Configuration
# ⚠️ CRITICAL: Class order MUST match production pipeline!
# Your pipeline (yolo_detector.py) expects: 0=mango, 1=orange, 2=guava, 3=grapefruit
yaml_content = {
    'path': str(dataset_path),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'nc': 4,
    'names': ['mango', 'orange', 'guava', 'grapefruit']  # ✅ CORRECTED ORDER
}

# Save yaml file to working directory (Kaggle allows writing here)
yaml_file = Path('/kaggle/working/data.yaml')
with open(yaml_file, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("✅ Created data.yaml configuration")
print(f"📄 YAML file: {yaml_file}")
print(f"Classes: {yaml_content['nc']}")
print(f"Names: {yaml_content['names']}")
print("⚠️  Class order MATCHES production pipeline (yolo_detector.py)")

# Display the created yaml content
print("\n📋 YAML Configuration:")
with open(yaml_file, 'r') as f:
    print(f.read())

In [ ]:
# Clean Dataset - Remove Corrupt Images and Validate Balance
import cv2
import numpy as np
import shutil
from collections import Counter

print("🧹 Cleaning dataset and validating class balance...")

# Create clean dataset directory
clean_dataset_path = Path('/kaggle/working/clean_dataset')
clean_dataset_path.mkdir(parents=True, exist_ok=True)

# Track class distribution and statistics
class_distribution = {'train': Counter(), 'val': Counter()}
class_names = ['mango', 'orange', 'guava', 'grapefruit']

for split in ['train', 'val']:
    print(f"\n{'='*60}")
    print(f"📁 Processing {split} split...")
    print('='*60)
    
    # Create directories
    (clean_dataset_path / split / 'images').mkdir(parents=True, exist_ok=True)
    (clean_dataset_path / split / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Get all images
    image_dir = dataset_path / split / 'images'
    label_dir = dataset_path / split / 'labels'
    
    if not image_dir.exists():
        print(f"   ❌ Directory not found: {image_dir}")
        continue
    
    images = list(image_dir.glob('*.jpg'))
    print(f"   📊 Found {len(images)} images to process")
    
    valid_count = 0
    corrupt_count = 0
    no_label_count = 0
    segmentation_cleaned_count = 0
    
    for idx, img_path in enumerate(images):
        try:
            # ✅ Basic image validation (cv2.imread is sufficient)
            img = cv2.imread(str(img_path))
            if img is None:
                raise ValueError("cv2.imread failed")
            
            if img.shape[0] < 10 or img.shape[1] < 10:
                raise ValueError("Image too small")
            
            # Get corresponding label file
            label_path = label_dir / (img_path.stem + '.txt')
            
            if not label_path.exists():
                no_label_count += 1
                continue
            
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            if not lines:
                no_label_count += 1
                continue
            
            # ✅ FIXED: Extract ONLY valid bounding boxes (5 values)
            # Skip segmentation lines (>5 values) instead of rejecting entire image
            valid_labels = []
            had_segmentation = False
            
            for line in lines:
                parts = line.strip().split()
                
                # ✅ CRITICAL: Accept EXACTLY 5 values (bounding box format)
                if len(parts) == 5:
                    try:
                        class_id = int(parts[0])
                        coords = [float(x) for x in parts[1:5]]
                        
                        # Validate ranges
                        if (0 <= class_id <= 3 and 
                            all(0.0 <= c <= 1.0 for c in coords) and
                            coords[2] > 0 and coords[3] > 0):
                            
                            valid_labels.append(f"{class_id} {coords[0]:.6f} {coords[1]:.6f} {coords[2]:.6f} {coords[3]:.6f}\n")
                            class_distribution[split][class_id] += 1
                    except (ValueError, IndexError):
                        continue
                elif len(parts) > 5:
                    # ✅ NEW: Try to extract first 5 values from segmentation format
                    # Some datasets store: class x y w h [polygon points...]
                    try:
                        class_id = int(parts[0])
                        coords = [float(x) for x in parts[1:5]]
                        
                        if (0 <= class_id <= 3 and 
                            all(0.0 <= c <= 1.0 for c in coords) and
                            coords[2] > 0 and coords[3] > 0):
                            
                            valid_labels.append(f"{class_id} {coords[0]:.6f} {coords[1]:.6f} {coords[2]:.6f} {coords[3]:.6f}\n")
                            class_distribution[split][class_id] += 1
                            had_segmentation = True
                    except (ValueError, IndexError):
                        continue
            
            if valid_labels:
                # Copy valid image
                shutil.copy(str(img_path), str(clean_dataset_path / split / 'images' / img_path.name))
                
                # Write cleaned labels (bounding boxes ONLY)
                with open(clean_dataset_path / split / 'labels' / label_path.name, 'w') as f:
                    f.writelines(valid_labels)
                
                valid_count += 1
                if had_segmentation:
                    segmentation_cleaned_count += 1
            else:
                no_label_count += 1
                
        except Exception as e:
            corrupt_count += 1
        
        # Progress indicator
        if (idx + 1) % 500 == 0:
            print(f"   Processed {idx + 1}/{len(images)}... (Valid: {valid_count}, Corrupt: {corrupt_count})")
    
    print(f"\n   📊 RESULTS:")
    print(f"      ✅ Valid images: {valid_count}")
    print(f"      ❌ Corrupt images: {corrupt_count}")
    print(f"      ⚠️  No valid labels: {no_label_count}")
    print(f"      🔧 Cleaned segmentation data: {segmentation_cleaned_count}")

# Display class distribution
print("\n" + "="*60)
print("📊 CLASS DISTRIBUTION AFTER CLEANING")
print("="*60)
for split in ['train', 'val']:
    print(f"\n{split.upper()} SET:")
    total = sum(class_distribution[split].values())
    if total == 0:
        print(f"  ❌ NO VALID IMAGES")
        continue
    for class_id in range(4):
        count = class_distribution[split][class_id]
        percentage = (count / total * 100) if total > 0 else 0
        print(f"  {class_names[class_id]:12s}: {count:5d} boxes ({percentage:5.1f}%)")
    print(f"  {'TOTAL BOXES':12s}: {total:5d}")

# Balance check
print("\n⚠️  BALANCE CHECK:")
for split in ['train', 'val']:
    counts = [class_distribution[split][i] for i in range(4)]
    if counts and max(counts) > 0:
        if min(counts) == 0:
            print(f"❌ {split.upper()}: Missing classes!")
        else:
            ratio = max(counts) / min(counts)
            if ratio > 3:
                print(f"⚠️  {split.upper()}: Imbalanced (ratio: {ratio:.1f}:1)")
            else:
                print(f"✅ {split.upper()}: Well balanced (ratio: {ratio:.1f}:1)")

# Verify cleaned dataset
train_images = len(list((clean_dataset_path / 'train/images').glob('*.jpg')))
val_images = len(list((clean_dataset_path / 'val/images').glob('*.jpg')))

print("\n" + "="*60)
print("🎯 CLEANED DATASET SUMMARY")
print("="*60)
print(f"   Train images: {train_images}")
print(f"   Val images: {val_images}")
print(f"   Total: {train_images + val_images}")

if train_images == 0 or val_images == 0:
    raise RuntimeError("Dataset cleaning failed - no valid images found")
else:
    print("\n✅ Dataset cleaning successful!")
    print("✅ Removed corrupt images and segmentation data")
    print("✅ Kept valid YOLO bounding boxes only")
    dataset_path = clean_dataset_path
    print(f"🔄 Using cleaned dataset: {dataset_path}")
    print("="*60)


In [ ]:
# Initialize YOLO Model and Train

# ✅ CRITICAL: Ensure we're using the cleaned dataset path
clean_dataset_path = Path('/kaggle/working/clean_dataset')

# ✅ RECREATE data.yaml with ABSOLUTE paths to cleaned dataset
yaml_content = {
    'path': str(clean_dataset_path),
    'train': str(clean_dataset_path / 'train/images'),  # ✅ ABSOLUTE PATH
    'val': str(clean_dataset_path / 'val/images'),      # ✅ ABSOLUTE PATH
    'nc': 4,
    'names': ['mango', 'orange', 'guava', 'grapefruit']
}

yaml_file = Path('/kaggle/working/data.yaml')
with open(yaml_file, 'w') as f:
    yaml.dump(yaml_content, f, default_flow_style=False)

print("📋 Updated YAML Configuration:")
with open(yaml_file, 'r') as f:
    print(f.read())

# Load model
model = YOLO('yolov8n.pt')
print("\n✅ YOLOv8 Nano model loaded successfully!")

# Train YOLO Model
print("\n🚀 Starting YOLO training on Kaggle...")
print("⏱️ Estimated time: 5-6 hours on P100 GPU")
print("📊 Training progress will be displayed below\n")

# Training with production-optimized parameters
results = model.train(
    data=str(yaml_file),
    epochs=150,
    imgsz=640,
    batch=32,
    name='yolo_detection',
    patience=20,
    save_period=10,
    device=0 if torch.cuda.is_available() else 'cpu',
    verbose=True,
    plots=True,
    workers=0,                   # ✅ CRITICAL FIX: Single-process to avoid OpenCV multiprocess issues
    project='/kaggle/working/runs/detect',
    
    # ✅ PERFORMANCE OPTIMIZATIONS
    amp=False,
    cache=False,
    
    # ✅ TRAINING STABILITY
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    warmup_momentum=0.8,
    
    # ✅ LOSS WEIGHTS
    box=7.5,
    cls=0.5,
    dfl=1.5,
    
    # ✅ REPRODUCIBILITY
    seed=42,
    deterministic=True,
    
    # ✅ VALIDATION & SAVING
    val=True,
    save=True,
    rect=False,
    resume=False,
    exist_ok=True,
    pretrained=True
)

print("\n🎉 Training completed!")
print("📁 Results: /kaggle/working/runs/detect/yolo_detection/")


In [ ]:
# Evaluate Model - Comprehensive Results
from IPython.display import Image, display
import matplotlib.pyplot as plt

# Load best model
best_model_path = '/kaggle/working/runs/detect/yolo_detection/weights/best.pt'
best_model = YOLO(best_model_path)
print("✅ Best model loaded\n")

# Evaluate on validation set
metrics = best_model.val()

# Display comprehensive performance metrics
print("="*60)
print("📊 MODEL PERFORMANCE METRICS")
print("="*60)
print(f"mAP@0.5:        {metrics.box.map50:.4f} {'✅ Excellent' if metrics.box.map50 > 0.85 else '⚠️ Needs improvement' if metrics.box.map50 > 0.70 else '❌ Poor'}")
print(f"mAP@0.5-0.95:   {metrics.box.map:.4f} {'✅ Excellent' if metrics.box.map > 0.70 else '⚠️ Good' if metrics.box.map > 0.50 else '❌ Needs improvement'}")
print(f"Precision:      {metrics.box.mp:.4f} {'✅ High' if metrics.box.mp > 0.85 else '⚠️ Moderate' if metrics.box.mp > 0.70 else '❌ Low'}")
print(f"Recall:         {metrics.box.mr:.4f} {'✅ High' if metrics.box.mr > 0.85 else '⚠️ Moderate' if metrics.box.mr > 0.70 else '❌ Low'}")
print("="*60)

# Performance assessment
print("\n🎯 TRAINING QUALITY ASSESSMENT:")
if metrics.box.map50 > 0.85 and metrics.box.map > 0.65:
    print("✅ Model is WELL-TRAINED and ready for deployment!")
elif metrics.box.map50 > 0.70:
    print("⚠️ Model shows GOOD performance but could be improved")
else:
    print("❌ Model needs MORE TRAINING or data improvement")

# Display per-class metrics (CORRECTED ORDER)
print("\n📋 PER-CLASS PERFORMANCE:")
class_names = ['mango', 'orange', 'guava', 'grapefruit']  # ✅ CORRECTED
if hasattr(metrics.box, 'ap50'):
    for i, name in enumerate(class_names):
        if i < len(metrics.box.ap50):
            ap = metrics.box.ap50[i]
            status = "✅" if ap > 0.85 else "⚠️" if ap > 0.70 else "❌"
            print(f"{status} {name:12s}: {ap:.4f}")

# Display all available training visualizations
print("\n📈 TRAINING VISUALIZATIONS:")
results_dir = '/kaggle/working/runs/detect/yolo_detection'

# 1. Training curves (Loss, mAP, Precision, Recall)
if os.path.exists(f'{results_dir}/results.png'):
    print("\n1️⃣ Training Curves (Loss, Metrics over Epochs):")
    display(Image(f'{results_dir}/results.png'))

# 2. Confusion Matrix
if os.path.exists(f'{results_dir}/confusion_matrix.png'):
    print("\n2️⃣ Confusion Matrix:")
    display(Image(f'{results_dir}/confusion_matrix.png'))

# 3. F1 Curve
if os.path.exists(f'{results_dir}/F1_curve.png'):
    print("\n3️⃣ F1-Confidence Curve:")
    display(Image(f'{results_dir}/F1_curve.png'))

# 4. Precision-Recall Curve
if os.path.exists(f'{results_dir}/PR_curve.png'):
    print("\n4️⃣ Precision-Recall Curve:")
    display(Image(f'{results_dir}/PR_curve.png'))

# 5. Validation Batch Labels
if os.path.exists(f'{results_dir}/val_batch0_labels.jpg'):
    print("\n5️⃣ Validation Batch - Ground Truth Labels:")
    display(Image(f'{results_dir}/val_batch0_labels.jpg'))

# 6. Validation Batch Predictions
if os.path.exists(f'{results_dir}/val_batch0_pred.jpg'):
    print("\n6️⃣ Validation Batch - Model Predictions:")
    display(Image(f'{results_dir}/val_batch0_pred.jpg'))

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE - Review the metrics and visualizations above")
print("📁 Download trained model from: /kaggle/working/runs/detect/multi_fruit_detection_v1/weights/")
print("="*60)

In [ ]:
# Download Model Files - Production Ready
import shutil

# Copy important files to output directory for easy download
output_dir = '/kaggle/working/model_output'
os.makedirs(output_dir, exist_ok=True)

# ✅ CRITICAL: Rename to match production pipeline
if os.path.exists(f'{results_dir}/weights/best.pt'):
    shutil.copy(
        f'{results_dir}/weights/best.pt',
        f'{output_dir}/yolo_detection_best.pt'  # ✅ PRODUCTION FILENAME
    )
    print("✅ Best model → yolo_detection_best.pt (ready for deployment)")
    
    # Also save last checkpoint
    if os.path.exists(f'{results_dir}/weights/last.pt'):
        shutil.copy(f'{results_dir}/weights/last.pt', f'{output_dir}/yolo_detection_last.pt')
        print("✅ Last checkpoint → yolo_detection_last.pt")

# Copy visualizations
for filename in ['results.png', 'confusion_matrix.png', 'F1_curve.png', 'PR_curve.png']:
    src = f'{results_dir}/{filename}'
    if os.path.exists(src):
        shutil.copy(src, f'{output_dir}/{filename}')
        print(f"✅ {filename} copied")

print(f"\n📁 Output files: {output_dir}")

# Final summary
print("\n" + "="*70)
print("🎯 TRAINING COMPLETE SUMMARY")
print("="*70)
print(f"   Model: YOLOv8 Nano")
print(f"   Classes: {class_names}")
print(f"   ✅ Class order matches production pipeline")
print(f"   Final mAP@0.5: {metrics.box.map50:.4f}")
print(f"   Final mAP@0.5-0.95: {metrics.box.map:.4f}")
print(f"   Platform: Kaggle")
print(f"   Status: ✅ Complete & Deployment-Ready")
print("="*70)

print("\n📋 DEPLOYMENT STEPS:")
print("1. Download 'yolo_detection_best.pt' from model_output folder")
print("2. Upload to your project: models/yolo_detection_best.pt")
print("3. Pipeline will auto-load (pipeline_config.py line 28)")
print("4. Test via Swagger: POST /api/detection/fruit")
print("\n✨ Model ready for production use!")
print("="*70)